<h1 style="color:#004080; font-size:2em; margin-bottom:0.3em;">
  S3 Read: Analyze a Cloud-Native IVT Zarr for Atmospheric Rivers
</h1>
<h4 style="color:gray; font-weight:normal; line-height:1.5;">
  This notebook shows how to authenticate with HydroShare and stream a cloud-native Zarr (~40&nbsp;GB) of Integrated Vapor Transport (IVT) derived from the CESM2 Large Ensemble (LENS2). The full collection (~1.9&nbsp;PB, NetCDF) is hosted by the NSF Research Data Archive (dataset d651056). In August&nbsp;2024, we extracted a ~40&nbsp;GB IVT subset from the legacy NCAR archive and republished it on HydroShare in Zarr format. Using HydroShare’s S3-compatible endpoint with xarray+dask, we perform time/space subsetting and detect atmospheric rivers (ARs) with the common IVT&nbsp;&ge;&nbsp;250&nbsp;kg/m/s&nbsp; threshold, then assign AR categories based on joint intensity and duration. We also compute an “AR degree” metric, defined as event-integrated IVT (the sum of IVT over an event’s duration; IVT × days), as a simple indicator of event severity.
</h4>




<div style="background-color: #f7f7f7; padding: 10px; border-radius: 5px;">
    <p><strong>Authors:</strong> 
    <ul>
        <li>Irene Garousi-Nejad, <a href="mailto:igarousi@cuahsi.org">igarousi@cuahsi.org</a></li>
        <li>Anthony Castronova, <a href="mailto:acastronova@cuahsi.org">acastronova@cuahsi.org</a></li>
    </ul>
    </p>
    <p><strong>Last Modified:</strong> 8/22/25</p>
    <p><strong>Software Requirements:</strong> This notebook has been tested using Python v3.11.8 using the following library versions:
        <ul>
            <li>dask>=2025.7.0</li>
            <li>xarray>=2024.2.0</li>
            <li>geopandas>=1.1.1</li>
            <li>h5netcdf>=1.6.4</li>
            <li>hvplot>=0.11.3</li>
            <li>requests>=2.32.2</li>
            <li>rioxarray>=0.19.0</li>
            <li>s3fs>=2025.5.1</li>
            <li>numpy>=2.3.1</li>
            <li>cftime>=1.6.4</li>
            <li>folium>=0.20.0</li>
        </ul>
    </p>
    <p><strong>Supporting Files and Dependencies:</strong> 
        <ul>
            <li>ARUtils.py</li>           
        </ul>
    </p>
    <p><strong>References:</strong></p>
    <ul>
        <li>https://psl.noaa.gov/technology/aro/</li>
        <li>https://cw3e.ucsd.edu/heavy-rain-in-the-pacific-northwest/</li>
        <li>https://rda.ucar.edu/datasets/d651056/dataaccess/#</li>
        <li>Ralph FM, Rutz JJ, Cordeira JM, Dettinger M, Anderson M, Reynolds D, Schick LJ, Smallcomb C (2019) A scale to characterize the Strength and impacts of Atmospheric Rivers. Bull Am Meteorol Soc 100(2):269–289. https://doi.org/10.1175/BAMS-D-18-0023.1</li>
        <li>https://www.cesm.ucar.edu/community-projects/lens2</li>
    </ul>
</div>

## 1. Environment Setup: Imports & Dask (Cluster/Client)

In [1]:
import importlib.util

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import dask
import folium
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import xarray as xr
from dask.distributed import Client

import ARUtils
import S3hsclient as hsclient

import zarr # noqa: F401 - ignore unused import warning for implicit dependency

This notebook requires several additional libraries to be installed. Since these are not directly imported above, we'll check to make sure they're installed on the system.

In [2]:
def is_installed(package_name):
    return importlib.util.find_spec(package_name) is not None

print("h5netcdf installed:", is_installed("h5netcdf"))
print("h5py installed:", is_installed("h5py"))
print("zarr installed:", is_installed("zarr"))

h5netcdf installed: True
h5py installed: True
zarr installed: True


Set up a local Dask cluster so we can run computations in parallel. This will greatly speed up our analysis. Note, the parameters `n_workers`, `threads_per_worker`, and `memory_limit` should be modified to meet the hardware in which this notebook will utilize.

In [3]:
# use a try accept loop so we only instantiate the client
# if it doesn't already exist.
try:
    print(client.dashboard_link)
except:
    # The client should be customized to your workstation resources.
    # This is configured for a "Large" instance on ciroh.awi.2i2c.cloud
    # client = Client()
    client = Client(n_workers=1, threads_per_worker=2, memory_limit="12GB")
    print(client.dashboard_link)

dask.config.set({"array.slicing.split_large_chunks": True})

http://127.0.0.1:8787/status


## 2. Authenticate to HydroShare & Configure S3FS 

We'll use the `S3hsclient` utility to obtain S3 credentials for accessing our data. This utility extends the HydroShare Python library , knowan as `hsclient`, to provide direct access to data stored in S3-compatible bucket storage. 


In [4]:
# sign in to HydroShare using the S3hsclient utility.
# This will ask you for your username and password.
hs = hsclient.S3HydroShare()

Username:  TonyCastronova
Password for TonyCastronova:  ········


Load the resource containing our data from in HydroShare:https://hydroshare.org/resource/53effa6f532b43a5972daa5336932224

In [5]:
resource_id = "53effa6f532b43a5972daa5336932224"
resource = hs.resource(resource_id)

Using the `S3hsclient` utility, list the files that exist in the root of this directory. Files can be listed for subdirectories by altering the `prefix` argument.

In [6]:
resource.s3_ls(prefix="")

['igarousi/53effa6f532b43a5972daa5336932224/data/contents/ivt.zarr']

## 3. Open the IVT Zarr 

Open lazily using `xarray`. The zarr dataset we're interested in is the first object listed by the `resource.s3_ls()` command above. Using this path, we can create an s3 mapper instance then load it using the xarray `open_zarr` function.

In [7]:
# get the S3 filesystem object
s3 = hs.get_s3_filesystem()

# get the path to the zarr store in HydroShare
zarr_data_path = resource.s3_ls()[0]

# initialize a mapper instance with the zarr path
mapper = s3.get_mapper(zarr_data_path)

# lazily load the zarr dataset into memory
ds = xr.open_zarr(mapper, consolidated=True)

# display the zarr metadata
ds

<xarray.Dataset> Size: 41GB
Dimensions:  (time: 366462, lat: 192, lon: 288)
Coordinates:
  * time     (time) object 3MB 1970-01-01 00:00:00 ... 2101-01-01 00:00:00
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
Data variables:
    IVT      (time, lat, lon) float16 41GB dask.array<chunksize=(1000, 192, 288), meta=np.ndarray>
Attributes:
    Conventions:       CF-1.0
    case:              b.e21.BSSP370smbb.f09_g17.LE2-1011.001
    host:              mom2
    initial_file:      b.e21.BHISTsmbb.f09_g17.LE2-1011.001.cam.i.2015-01-01-...
    logname:           sunseon
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    source:            CAM
    time_period_freq:  hour_6
    topography_file:   /mnt/lustre/share/CESM/cesm_input/atm/cam/topo/fv_0.9x...

In [8]:
ds.isel(time=1).IVT.plot()

2026-05-20 15:34:03,082 - distributed.worker - ERROR - Compute Failed
Key:       getitem-open_dataset-IVT-original-finalize-hlgfinalizecompute-90e6473768b244d9abbedff07e4055c6
State:     executing
Task:  <Task 'getitem-open_dataset-IVT-original-finalize-hlgfinalizecompute-90e6473768b244d9abbedff07e4055c6' _execute_subgraph(...)>
Exception: 'EndpointConnectionError(\'Could not connect to the endpoint URL: "https://s3.hydroshare.org/igarousi/53effa6f532b43a5972daa5336932224/data/contents/ivt.zarr/IVT/0.0.0"\')'
Traceback: '  File "/Users/castro/Documents/work/notebooks/Science Examples/Detecting Atmospheric Rivers from CESM2 LENS Data/.venv/lib/python3.11/site-packages/dask/array/core.py", line 141, in getter\n    c = np.asarray(c)\n        ^^^^^^^^^^^^^\n  File "/Users/castro/Documents/work/notebooks/Science Examples/Detecting Atmospheric Rivers from CESM2 LENS Data/.venv/lib/python3.11/site-packages/xarray/core/indexing.py", line 659, in __array__\n    return np.asarray(self.get_duck_a

EndpointConnectionError: Could not connect to the endpoint URL: "https://s3.hydroshare.org/igarousi/53effa6f532b43a5972daa5336932224/data/contents/ivt.zarr/IVT/0.0.0"

Check the dataset chunk sizes

In [ ]:
print(ds.IVT.chunksizes)

Check file size. Zarr chunks are compressed on disk. HydroShare shows the compressed total (e.g., 24 GiB). `ds.nbytes` reports the uncompressed size of all arrays in the dataset, i.e., what it would take in RAM if loaded.

In [ ]:
print(f"Variable size: {ds.nbytes/1e9:.1f} GB")
print("Number of years in one file:", round(ds.time.shape[0] / 4 / 365))

The time coordinate in this dataset is not strictly monotonic increasing, which can break label-based slicing, resampling, and merges. First, check it and then fix the order:

In [ ]:
def time_order_report(ds: xr.Dataset, dim: str = "time"):
    idx = ds.indexes.get(dim)
    if idx is None:
        raise KeyError(f"No index found for '{dim}'.")
    print(
        f"{dim}: monotonic↑={idx.is_monotonic_increasing}, "
        f"monotonic↓={idx.is_monotonic_decreasing}, "
        f"unique={idx.is_unique}, "
        f"length={len(idx)}"
    )
    return idx


# Example usage
idx = time_order_report(ds)
idx

In [ ]:
# If not strictly increasing or not unique, fix it:
ds = ds.sortby("time")

# Re-check
_ = time_order_report(ds)

Also, the time axis is a CFTimeIndex, i.e., it uses `cftime` date types for non-Gregorian calendars (e.g., noleap, 360_day). Many plotting and pandas/xarray functions work best with standard NumPy `datetimes`, so we convert to proleptic_gregorian (and drop cftime) before plotting:

In [ ]:
# Convert to proleptic_gregorian and drop cftime (requires modern xarray)
ds = ds.convert_calendar("proleptic_gregorian", use_cftime=False)

# If there are duplicate timestamps, drop them first (optional but helpful)
if ds.get_index("time").has_duplicates:
    ds = ds.sel(time=~ds.get_index("time").duplicated())

*Note*: This conversion will drop timestamps that do not exist in the Gregorian calendar (e.g., Feb 29 in noleap).

## 4. Subset to a Location and Period of Interest

Bodega Bay (38.3332° N, 123.0481° W) sits in California’s North Bay, a region frequently impacted by wintertime atmospheric rivers—major drivers of seasonal precipitation and flooding across the Pacific Northwest and Northern California (see [NOAA PSL’s Atmospheric River Observatory](https://psl.noaa.gov/technology/aro/)). In October 2014, for example, the Bodega Bay observatory documented the southernmost reach of an AR that brought significant rainfall to northwestern California ([CW3E report](https://cw3e.ucsd.edu/heavy-rain-in-the-pacific-northwest/)).

In this notebook, we focus on 1970–2009, a late-20th-century subset of the CESM2 LENS historical simulations (1850–2014).

In [ ]:
start_date = "1970-01-01"
end_date = "2009-12-31"
lat = 38.0
lon = -123.125

In [ ]:
m = folium.Map(location=[lat, lon], zoom_start=10)
folium.CircleMarker(
    [lat, lon],
    radius=6,
    color="red",
    fill=True,
    fill_opacity=0.8,
    popup=f"Lat {lat:.4f}, Lon {lon:.4f}",
).add_to(m)
m

In [ ]:
ds_point = (
    ds.sel(time=slice(start_date, end_date))
    .sel(lat=lat, lon=lon + 360.0, method="nearest")
    .IVT
)
ds_point

Load these data into memory and plot them.

In [ ]:
%%time
ds_point = ds_point.load()

In [ ]:
p = (
    ds_point.hvplot(label="IVT", color="black", line_dash="dashed", line_width=1)
).opts(width=900, height=400, show_grid=True, tools=["hover"])

p

Select a set of grid cells that cover Bodega Bay, CA (38.3332° N, 123.0481° W).

In [ ]:
# the nearest index for the lat/lon provided above
ilon = list(ds.lon.values).index(ds.sel(lon=lon + 360.0, method="nearest").lon)
ilat = list(ds.lat.values).index(ds.sel(lat=lat, method="nearest").lat)

cell_buffer = 10
# subset these data to include cells above and below this lat/lon
ds_box = (
    ds.sel(time=slice(start_date, end_date))
    .isel(
        lat=range(ilat - cell_buffer, ilat + cell_buffer),
        lon=range(ilon - cell_buffer, ilon + cell_buffer),
    )
    .IVT
)

ds_box = ds_box.to_dataset()

In [ ]:
%%time
ds_box = ds_box.load()

Plot the IVT data for these cells at time=0 to visualize our domain.

In [ ]:
fig, ax = plt.subplots()
ax.set_xticks([])
ax.set_yticks([])
ax = plt.axes(projection=ccrs.PlateCarree())

# add coastlines
ax.add_feature(cfeature.COASTLINE)

# add lat lon grids
gl = ax.gridlines(draw_labels=True, color="lightgrey", alpha=0.5, linestyle="--")
gl.xlabels_bottom = False
gl.ylabels_right = False

plot = ds_box.isel(time=0).IVT.plot(
    label="", add_labels=False, add_colorbar=False, xticks=None, yticks=None
)

# add custom colorbar inside the figure
cbar_ax = fig.add_axes(
    [0.99, 0.25, 0.02, 0.5]
)  # [left, bottom, width, height] in figure coords
cbar = plt.colorbar(plot, cax=cbar_ax)
cbar.set_label("IVT (kg m⁻¹ s⁻¹)")

ax.plot(
    lon + 360.0,
    lat,
    marker="o",
    color="red",
    markersize=8,
    transform=ccrs.PlateCarree(),
)

plt.show()

## 5. Identify Atmospheric River Events

To identify atmospheric rivers, we first flag periods where IVT ≥ 250 kg/m/s, a common screening threshold for AR conditions (Ralph et al., 2019). We then classify AR strength into Categories 1–5 using the Ralph et al. (2019) scale, which combines peak IVT and event duration. Lower categories indicate weaker ARs, while higher categories indicate extreme/exceptional events. This workflow is implemented in our Python utility module `ARUtils.py`.

In [ ]:
ds_box

In [ ]:
ds_box.chunks

In [ ]:
# # Ensure time is a single chunk (so events aren’t broken across chunks)
# ds_box = ds_box.chunk({"time": -1})
# ds_box.chunks

Assign an index to each group of timesteps that meet AR conditions

In [ ]:
ds_box["AR_INDEX"] = xr.apply_ufunc(
    ARUtils.identify_ar_events,
    ds_box.IVT,
    input_core_dims=[["time"]],
    output_core_dims=[["time"]],
    vectorize=True,
    dask="parallelized",
    dask_gufunc_kwargs={"allow_rechunk": True},
)

Compute the duration for each AR event

In [ ]:
ds_box["AR_DURATION"] = xr.apply_ufunc(
    ARUtils.compute_ar_durations,
    ds_box.AR_INDEX,
    input_core_dims=[["time"]],
    output_core_dims=[["time"]],
    vectorize=True,
    dask="parallelized",
    dask_gufunc_kwargs={"allow_rechunk": True},
)

Compute the category (1-5) for each AR event

In [ ]:
ds_box["AR_CATEGORY"] = xr.apply_ufunc(
    ARUtils.compute_ar_category,
    ds_box.AR_INDEX,
    ds_box.AR_DURATION,
    ds_box.IVT,
    input_core_dims=[["time"], ["time"], ["time"]],
    output_core_dims=[["time"]],
    vectorize=True,
    dask="parallelized",
    dask_gufunc_kwargs={"allow_rechunk": True},
)

Compute the probability of each AR event occurring

In [ ]:
res = xr.apply_ufunc(
    ARUtils.compute_ar_probability,
    ds_box.AR_INDEX,
    ds_box.AR_CATEGORY,
    input_core_dims=[["time"], ["time"]],
    output_core_dims=[["category"]],
    # output_sizes={'category': 5},
    output_dtypes=[float],
    vectorize=True,
    dask="parallelized",
    dask_gufunc_kwargs={"allow_rechunk": True, "output_sizes": {"category": 5}},
)

# Add the category dimension (from 1 to 5)
res = res.assign_coords(category=np.arange(1, 6))

# add the newly computed probabilty back to the original dataset
ds = ds.assign(AR_PROBABILITY=res)

Compute the magnitude of each AR event.

In [ ]:
ds_box["AR_DEGREE"] = xr.apply_ufunc(
    ARUtils.compute_ar_degree,
    ds_box.IVT,
    ds_box.AR_DURATION,
    ds_box.AR_INDEX,
    input_core_dims=[["time"], ["time"], ["time"]],
    output_core_dims=[["time"]],
    output_dtypes=[float],
    vectorize=True,
    dask="parallelized",
    dask_gufunc_kwargs={"allow_rechunk": True},
)

## 6. Plot Computed AR Category

Mean Atmospheric Rivers category through the record of data. Let's first build contour plots of mean AR_CATEGORY for each decade in our dataset.

In [ ]:
year = ds_box["time"].dt.year
decade = np.floor(year / 10) * 10
ds_box = ds_box.assign_coords({"decade": decade})

In [ ]:
ds_decade = (
    ds_box.where(ds_box.AR_CATEGORY > 0)
    .AR_CATEGORY.groupby("decade")
    .mean(dim="time")
    .compute()
)

In [ ]:
fig, ax = plt.subplots(
    nrows=2,
    ncols=2,
    subplot_kw={"projection": ccrs.PlateCarree()},
    figsize=(10, 8),
    layout="constrained",
)

axes = ax.flatten()
for i in range(0, len(ds_decade.decade)):
    axes[i].coastlines()
    axes[i].gridlines(draw_labels=True, color="lightgrey", alpha=0.5, linestyle="--")

    # created grid and contour plots.
    d_plot = ds_decade.isel(decade=i).plot(
        label="", add_labels=False, add_colorbar=False, ax=axes[i]
    )

    c_plot = ds_decade.isel(decade=i).plot.contour(
        add_labels=True, add_colorbar=False, colors="white", ax=axes[i]
    )
    axes[i].clabel(c_plot)

    st = int(ds_decade.decade[i].item())
    et = st + 9
    axes[i].set_title(f"{st} - {et}")

fig.colorbar(d_plot, ax=ax[:, 1], shrink=0.8)
fig.suptitle("Mean Decadal AR Category");

Build contour plots of mean AR_DEGREE for each decade in our dataset.

In [ ]:
ds_decade = (
    ds_box.where(ds_box.AR_CATEGORY > 0)
    .AR_DEGREE.groupby("decade")
    .mean(dim="time")
    .compute()
)

In [ ]:
fig, ax = plt.subplots(
    nrows=2,
    ncols=2,
    subplot_kw={"projection": ccrs.PlateCarree()},
    figsize=(10, 8),
    layout="constrained",
)

axes = ax.flatten()
for i in range(0, len(ds_decade.decade)):
    axes[i].coastlines()
    axes[i].gridlines(draw_labels=True, color="lightgrey", alpha=0.5, linestyle="--")

    # created grid and contour plots.
    d_plot = ds_decade.isel(decade=i).plot(
        label="", add_labels=False, add_colorbar=False, ax=axes[i]
    )

    c_plot = ds_decade.isel(decade=i).plot.contour(
        add_labels=True, add_colorbar=False, colors="white", ax=axes[i]
    )
    fmt = ticker.ScalarFormatter(useMathText=True)
    fmt.set_powerlimits((0, 0))
    fmt.create_dummy_axis()

    def fmt_func(value):
        return f"{value:.0e}"

    axes[i].clabel(c_plot, inline=True, fontsize=10, fmt=fmt_func)

    st = int(ds_decade.decade[i].item())
    et = st + 9
    axes[i].set_title(f"{st} - {et}")

fig.colorbar(d_plot, ax=ax[:, 1], shrink=0.8)
fig.suptitle("Mean Decadal AR Degree");